# Evaluador común
El puntaje por imagen conserva el campo de visión nativo. Para la comparación por píxel, todos los mapas y máscaras se proyectan a la región central que ve Dinomaly (recorte de 392 × 392 sobre una cuadrícula de 448 × 448, tomados de `configs/models/dinomaly.json`). Se informa cuántos defectos quedan fuera de esa región.

In [ ]:
from scipy import ndimage
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)
import torch.nn.functional as F


COMMON = json.loads((ROOT / "configs/models/dinomaly.json").read_text())
COMMON_GRID, COMMON_CROP = COMMON["resize"], COMMON["crop"]
COMMON_OFFSET = (COMMON_GRID - COMMON_CROP) // 2


def common_map(tensor, cropped, mode):
    size = COMMON_CROP if cropped else COMMON_GRID
    options = {"align_corners": False} if mode == "bilinear" else {}
    grid = F.interpolate(
        tensor.float().reshape(1, 1, *tensor.shape[-2:]),
        size=(size, size),
        mode=mode,
        **options,
    )[0, 0]
    end = COMMON_OFFSET + COMMON_CROP
    return (grid if cropped else grid[COMMON_OFFSET:end, COMMON_OFFSET:end]).cpu().numpy()

## Predicciones y latencia en GPU

In [ ]:
@torch.inference_mode()
def predict_rows(model, transform, frame, cfg, folder, maps):
    folder.mkdir(parents=True, exist_ok=True)
    samples = Samples(frame, transform)
    rows = []
    model.eval().cuda()
    for i in range(len(samples)):
        image, native_mask, label, path = samples[i]
        batch = image.unsqueeze(0).cuda()
        with torch.autocast("cuda", enabled=cfg["precision"] == "16-mixed"):
            if i == 0:
                for _ in range(10):
                    model(batch)
            torch.cuda.synchronize()
            start = time.perf_counter()
            prediction = model(batch)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
        score = float(prediction.pred_score.item())
        if not np.isfinite(score):
            raise ValueError("Puntuación de imagen no finita.")
        rows.append(dict(path=path, label=label, score=score, seconds=elapsed))
        if maps:
            if folder.name == "public-test":
                mask = native_mask.cpu().numpy().astype(bool)
                anomaly = (
                    F.interpolate(
                        prediction.anomaly_map.float().reshape(
                            1, 1, *prediction.anomaly_map.shape[-2:]
                        ),
                        size=mask.shape,
                        mode="bilinear",
                        align_corners=False,
                    )[0, 0]
                    .cpu()
                    .numpy()
                )
            else:
                anomaly = common_map(
                    prediction.anomaly_map, bool(cfg["crop"]), "bilinear"
                )
                original_mask = samples.rows[i].mask
                if original_mask is None:
                    mask = np.zeros(anomaly.shape, dtype=bool)
                else:
                    with Image.open(original_mask) as source:
                        mask = (
                            common_map(
                                pil_to_tensor(source.convert("L")),
                                False,
                                "nearest-exact",
                            )
                            > 0
                        )
            if not np.isfinite(anomaly).all():
                raise ValueError("Mapa de anomalías no finito.")
            np.savez_compressed(folder / f"{i:06d}.npz", score=anomaly, mask=mask)
    results = pd.DataFrame(rows)
    results.to_csv(folder / "predictions.csv", index=False)
    return results

## Integración de curvas por píxel y por región

In [ ]:
def pixel_metrics(folder):
    paths = sorted(folder.glob("*.npz"))
    low, high = np.inf, -np.inf
    for path in paths:
        with np.load(path) as data:
            scores = data["score"]
            low, high = min(low, float(scores.min())), max(high, float(scores.max()))
    bins = 65536
    positives, negatives, overlaps = np.zeros((3, bins))
    regions = 0
    positive_images = 0
    for path in paths:
        with np.load(path) as data:
            mask, scores = data["mask"], data["score"].astype(np.float64)
        levels = (
            ((scores - low) / max(high - low, 1e-15) * (bins - 1))
            .astype(int)
            .clip(0, bins - 1)
        )
        positives += np.bincount(levels[mask], minlength=bins)
        negatives += np.bincount(levels[~mask], minlength=bins)
        labels, count = ndimage.label(mask, structure=np.ones((3, 3)))
        if count:
            positive_images += 1
            areas = np.bincount(labels[mask])
            overlaps += np.bincount(
                levels[mask], weights=1 / areas[labels[mask]], minlength=bins
            )
            regions += count
    if not regions or not negatives.sum():
        raise ValueError("La región evaluada necesita píxeles positivos y negativos.")

    def cumulative(hist):
        return np.r_[0.0, hist[::-1].cumsum()]

    tp, fp = cumulative(positives), cumulative(negatives)
    recall, fpr, pro = tp / tp[-1], fp / fp[-1], cumulative(overlaps) / regions
    precision = np.divide(tp, tp + fp, out=np.ones_like(tp), where=(tp + fp) > 0)
    limit = 0.30
    keep = fpr <= limit
    partial = (
        np.trapezoid(
            np.r_[pro[keep], np.interp(limit, fpr, pro)], np.r_[fpr[keep], limit]
        )
        / limit
    )
    return dict(
        pixel_auroc=float(np.trapezoid(recall, fpr)),
        pixel_ap=float(sum(np.diff(recall) * precision[1:])),
        au_pro=float(partial),
        visible_positive_images=positive_images,
        regions=regions,
    )

## Métricas de evaluación

In [ ]:
def assess(model, transform, frame, cfg, folder, threshold):
    torch.cuda.reset_peak_memory_stats()
    results = predict_rows(model, transform, frame, cfg, folder, maps=True)
    label, score = results.label, results.score
    chosen = score >= threshold
    metrics = pixel_metrics(folder)
    metrics.update(
        image_auroc=roc_auc_score(label, score),
        image_ap=average_precision_score(label, score),
        f1=f1_score(label, chosen, zero_division=0),
        precision=precision_score(label, chosen, zero_division=0),
        recall=recall_score(label, chosen, zero_division=0),
        threshold=threshold,
        latency_p50_ms=1000 * results.seconds.quantile(0.5),
        latency_p95_ms=1000 * results.seconds.quantile(0.95),
        images_per_second=1 / results.seconds.mean(),
        images=len(results),
        outside_roi=int(label.sum()) - metrics["visible_positive_images"],
        inference_peak_gib=torch.cuda.max_memory_allocated() / 2**30,
    )
    save_json(folder / "metrics.json", metrics)
    return metrics

## Calibración con VAL

In [ ]:
def evaluate_validation(model, transform, frame, cfg, run, stage):
    valid = predict_rows(
        model, transform, frame[frame.split == "valid"], cfg, run / "valid", maps=False
    )
    if valid.label.nunique() == 2:
        save_json(run / "valid" / "metrics.json", {
            "image_auroc": roc_auc_score(valid.label, valid.score),
            "image_ap": average_precision_score(valid.label, valid.score),
        })
    if stage == "benchmark":
        threshold = float(np.quantile(valid.score, 0.95))
    else:
        precision, recall, thresholds = precision_recall_curve(valid.label, valid.score)
        f1 = (
            2
            * precision[:-1]
            * recall[:-1]
            / np.maximum(precision[:-1] + recall[:-1], 1e-15)
        )
        threshold = float(thresholds[np.argmax(f1)])
    save_json(run / "threshold.json", {"source": "valid", "value": threshold})
    if stage == "benchmark":
        assess(
            model,
            transform,
            frame[frame.split == "test"],
            cfg,
            run / "public-test",
            threshold,
        )